In [0]:
%sql
select b.ndc11, count(distinct a.patient_id) from cmpa_insights_internal_schema.patient360_master a
left join com_intgr.claims_pharmacy_events b
on a.patient_id = b.PATIENT_ID
where b.ndc11 = 8497600101 
GROUP BY b.ndc11

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
%sql
select * from cmpa_insights_internal_schema.patient360_master;

In [0]:
%sql
-- =============================================================================
-- patient360_master (Final patient-level master table)
--
-- What this step does:
--   Materializes a single, “wide” patient-level table by combining three upstream
--   datasets into one row per patient:
--     A) patient360_base              -> core demographics + claim-derived milestones + HCP features
--     B) most_recently_treated_hcp    -> most recent treating HCP in refresh window + visit context metrics
--     C) primary_hcp                  -> primary HCP assignment + HCO/territory/region enrichment
--
-- Output:
--   com_edp_prd.cmpa_insights_internal_schema.patient360_master
--
-- Join strategy:
--   - Start from patient360_base (a) as the backbone (all patients retained)
--   - LEFT JOIN most_recently_treated_hcp (b) on patient_id to add recent-treatment attribution fields
--   - LEFT JOIN primary_hcp (c) on patient_id to add primary HCP and territory enrichment fields
--   - SELECT DISTINCT used to dedupe in case joins introduce multiplicity (e.g., enrichment tables
--     or upstream views have >1 row per patient)
--
-- Column handling:
--   - b.* EXCEPT(patient_id) and c.* EXCEPT(patient_id) avoid duplicating patient_id columns
--     from the joined datasets.
--   - Window suffixes (_2yr/_3yr/_5yr) indicate derivation windows from upstream logic.
--
-- Parameters:
--   None directly here (all parameterized logic occurs upstream), but this depends on upstream
--   tables/views that may be parameterized by ${end_date}.
-- =============================================================================

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
SELECT DISTINCT
    -- -------------------------------------------------------------------------
    -- Patient identity + demographics (from patient360_base)
    -- -------------------------------------------------------------------------
    a.PATIENT_ID,
    a.PATIENT_YOB,
    a.PATIENT_AGE,
    a.PATIENT_GENDER,
    a.patient_state,

    -- -------------------------------------------------------------------------
    -- Historical milestones (patient-level; timeframe-agnostic where defined upstream)
    --   - incidence_date: earliest observed Dx date across history (as defined in base)
    --   - first_incidence_treatment_date: earliest observed treatment date across history (as defined in base)
    -- -------------------------------------------------------------------------
    a.incidence_date,
    a.first_incidence_treatment_date,

    -- -------------------------------------------------------------------------
    -- Latest claim (patient-level)
    --   - latest_claim_date may fall back to a patient-level date if NPI attribution is missing upstream
    -- -------------------------------------------------------------------------
    a.latest_claim_date,

    -- -------------------------------------------------------------------------
    -- Latest claim attributed HCP + visit context (from patient360_base)
    --   Note: these fields are populated only if the latest claim had an attributable NPI upstream
    -- -------------------------------------------------------------------------
    a.latest_claim_hcp_npi,
    a.latest_claim_hcp_name,
    a.latest_claim_hcp_specialty,
    a.latest_claim_hcp_visit_count,

    -- Latest claim attributed HCO (via reference enrichment in base)
    a.latest_claim_hcp_hco_npi,
    a.latest_claim_hcp_hco_name,

    -- -------------------------------------------------------------------------
    -- Latest treatment (patient-level) + derived treatment type
    --   - latest_treatment_date may fall back to a patient-level date if NPI attribution is missing upstream
    --   - latest_mpsii_tx_type typically derived within the refresh window upstream
    -- -------------------------------------------------------------------------
    a.latest_treatment_date,
    a.latest_mpsii_tx_type,

    -- -------------------------------------------------------------------------
    -- First Tx after Dx + timelines (from patient360_base)
    --   - first_tx_after_diagnosis is computed upstream (all-time tx constrained to >= incidence_date)
    --   - timelines are derived from incidence_date / first_tx_after_diagnosis / latest_treatment_date
    -- -------------------------------------------------------------------------
    a.first_tx_after_diagnosis,
    a.time_dx_to_first_tx_in_months,
    a.treatment_period_months,

    -- -------------------------------------------------------------------------
    -- Treatment activity proxy (from patient360_base)
    --   - elaprase_fills: distinct tx dates in refresh window as defined upstream
    -- -------------------------------------------------------------------------
    a.elaprase_fills,

    -- -------------------------------------------------------------------------
    -- Latest treatment attributed HCP + visit context (from patient360_base)
    -- -------------------------------------------------------------------------
    a.latest_treatment_hcp_npi,
    a.latest_treatment_hcp_name,
    a.latest_treatment_hcp_specialty,
    a.latest_treatment_hcp_visit_count,

    -- Latest treatment attributed HCO (via reference enrichment in base)
    a.latest_treatment_hcp_hco_npi,
    a.latest_treatment_hcp_hco_name,

    -- -------------------------------------------------------------------------
    -- First Dx / Tx attributed HCP metrics (5y-ish universe; from patient360_base)
    -- -------------------------------------------------------------------------
    a.first_dx_hcp_5yr,
    a.first_dx_all_visit_count_5yr,
    a.first_dx_last_visit_5yr,
    a.first_tx_hcp_5yr,
    a.first_tx_all_visit_count_5yr,
    a.first_tx_treatment_visit_count_5yr,
    a.first_tx_last_visit_5yr,

    -- -------------------------------------------------------------------------
    -- Top-5 most-seen HCPs (ranked by 3y; includes 5y counts + last-visit; from patient360_base)
    -- -------------------------------------------------------------------------
    a.most_seen_hcp1_3yr_ranked,
    a.most_seen_hcp1_visit_count_5yr,
    a.most_seen_hcp1_last_visit_5yr,
    a.most_seen_hcp2_3yr_ranked,
    a.most_seen_hcp2_visit_count_5yr,
    a.most_seen_hcp2_last_visit_5yr,
    a.most_seen_hcp3_3yr_ranked,
    a.most_seen_hcp3_visit_count_5yr,
    a.most_seen_hcp3_last_visit_5yr,
    a.most_seen_hcp4_3yr_ranked,
    a.most_seen_hcp4_visit_count_5yr,
    a.most_seen_hcp4_last_visit_5yr,
    a.most_seen_hcp5_3yr_ranked,
    a.most_seen_hcp5_visit_count_5yr,
    a.most_seen_hcp5_last_visit_5yr,

    -- -------------------------------------------------------------------------
    -- Add-on enrichment block #1: most_recently_treated_hcp
    --   Pulls in:
    --     - most_recently_treated_hcp_2yr and its attributes (name/specialty/HCO/territory/region)
    --     - visit context metrics computed over the broader claims universe in that view
    --   EXCEPT(patient_id) avoids duplicating the patient id column.
    -- -------------------------------------------------------------------------
    b.* EXCEPT (patient_id),

    -- -------------------------------------------------------------------------
    -- Add-on enrichment block #2: primary_hcp
    --   Pulls in:
    --     - primary HCP attribution + name/HCO/territory/region fields
    --   EXCEPT(patient_id) avoids duplicating the patient id column.
    -- -------------------------------------------------------------------------
    c.* EXCEPT (patient_id),
    -- -------------------------------------------------------------------------
-- Add-on enrichment block #3: TIVI HCP
-- -------------------------------------------------------------------------

    d.tivi_first_tx_hcp_5yr,
    d.tivi_first_tx_visit_count_5yr,

    d.tivi_latest_tx_hcp,
    d.tivi_latest_tx_visit_count_5yr,

    d.tivi_most_seen_hcp1,
    d.tivi_most_seen_hcp2,
    d.tivi_most_seen_hcp3,
    d.tivi_most_seen_hcp4,
    d.tivi_most_seen_hcp5,
    -- =========================
    -- TIVI: LATEST HCP ENRICHMENT
    -- =========================
    CONCAT(pdtivi_latest_tx.FIRST_NAME, ' ', pdtivi_latest_tx.LAST_NAME) AS tivi_latest_tx_hcp_name,
    pdtivi_latest_tx.PRIMARY_SPECIALTY AS tivi_latest_tx_hcp_specialty,
    ref_tivi_latest_tx.hco_npi         AS tivi_latest_tx_hcp_hco_npi,
    ref_tivi_latest_tx.hco_name        AS tivi_latest_tx_hcp_hco_name,

    -- =========================
    -- TIVI: FIRST HCP ENRICHMENT
    -- =========================
    CONCAT(pdtivi_first_tx.FIRST_NAME, ' ', pdtivi_first_tx.LAST_NAME) AS tivi_first_tx_hcp_name,
    pdtivi_first_tx.PRIMARY_SPECIALTY  AS tivi_first_tx_hcp_specialty,
    ref_tivi_first_tx.hco_npi          AS tivi_first_tx_hcp_hco_npi,
    ref_tivi_first_tx.hco_name         AS tivi_first_tx_hcp_hco_name,

    -- =========================
-- TIVI MOST SEEN HCP 1
-- =========================
CONCAT(pdtivi_m1.FIRST_NAME, ' ', pdtivi_m1.LAST_NAME) AS tivi_most_seen_hcp1_name,
pdtivi_m1.PRIMARY_SPECIALTY AS tivi_most_seen_hcp1_specialty,
ref_tivi_m1.hco_name AS tivi_most_seen_hcp1_hco_name,

-- =========================
-- TIVI MOST SEEN HCP 2
-- =========================
CONCAT(pdtivi_m2.FIRST_NAME, ' ', pdtivi_m2.LAST_NAME) AS tivi_most_seen_hcp2_name,
pdtivi_m2.PRIMARY_SPECIALTY AS tivi_most_seen_hcp2_specialty,
ref_tivi_m2.hco_name AS tivi_most_seen_hcp2_hco_name

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base AS a

-- Left join retains all patients from patient360_base even if no match in most_recently_treated_hcp
LEFT JOIN most_recently_treated_hcp AS b
    ON a.PATIENT_ID = b.patient_id

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS c
    ON a.PATIENT_ID = c.patient_id

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_tivi_hcp_summary AS d
    ON a.PATIENT_ID = d.patient_id
LEFT JOIN com_raw.kom_providers pdtivi_latest_tx
    ON d.tivi_latest_tx_hcp = pdtivi_latest_tx.npi
   AND pdtivi_latest_tx.provider_type = 'INDIVIDUAL'

LEFT JOIN com_raw.kom_providers pdtivi_first_tx
    ON d.tivi_first_tx_hcp_5yr = pdtivi_first_tx.npi
   AND pdtivi_first_tx.provider_type = 'INDIVIDUAL'

LEFT JOIN cmpa_insights_internal_schema.reference_file_pooja_1703 ref_tivi_latest_tx
    ON d.tivi_latest_tx_hcp = ref_tivi_latest_tx.hcp_npi

LEFT JOIN cmpa_insights_internal_schema.reference_file_pooja_1703 ref_tivi_first_tx
    ON d.tivi_first_tx_hcp_5yr = ref_tivi_first_tx.hcp_npi
-- =========================
-- TIVI MOST SEEN HCP 1
-- =========================
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703 ref_tivi_m1
    ON d.tivi_most_seen_hcp1 = ref_tivi_m1.hcp_npi

LEFT JOIN com_raw.kom_providers pdtivi_m1
    ON d.tivi_most_seen_hcp1 = pdtivi_m1.npi
   AND pdtivi_m1.provider_type = 'INDIVIDUAL'

-- =========================
-- TIVI MOST SEEN HCP 2
-- =========================
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703 ref_tivi_m2
    ON d.tivi_most_seen_hcp2 = ref_tivi_m2.hcp_npi

LEFT JOIN com_raw.kom_providers pdtivi_m2
    ON d.tivi_most_seen_hcp2 = pdtivi_m2.npi
   AND pdtivi_m2.provider_type = 'INDIVIDUAL'

In [0]:
%sql
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_tivi_hcp_summary AS

WITH

-- ============================================
-- 1. ELIGIBLE PATIENTS (REUSE BASE TABLE)
-- ============================================

eligible_patients AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base
),

-- ============================================
-- 2. PROVIDER FILTER (SAME AS BASE)
-- ============================================

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

-- ============================================
-- 3. TIVI TX CLAIMS (5YR WINDOW)
-- ============================================

tivi_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 = '50383066730'
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 = '50383066730'
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    )
    WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      AND (npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL)
),

-- ============================================
-- 4. 3YR + 2YR WINDOWS
-- ============================================

tivi_tx_claims_3yr AS (
    SELECT *
    FROM tivi_tx_claims_5yr
    WHERE FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
),

tivi_tx_claims_2yr AS (
    SELECT *
    FROM tivi_tx_claims_5yr
    WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- ============================================
-- 5. FIRST TIVI HCP (5YR)
-- ============================================

tivi_first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE), NPI) rn
    FROM tivi_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

tivi_first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS tivi_first_tx_hcp_5yr, first_date
    FROM tivi_first_tx_hcp_ranked
    WHERE rn = 1
),

tivi_first_tx_stats AS (
    SELECT f.PATIENT_ID, f.tivi_first_tx_hcp_5yr,
           COUNT(DISTINCT t.FILL_DATE) AS visit_count_5yr,
           MAX(t.FILL_DATE) AS last_visit_5yr
    FROM tivi_first_tx_hcp f
    LEFT JOIN tivi_tx_claims_5yr t
      ON f.PATIENT_ID = t.PATIENT_ID AND f.tivi_first_tx_hcp_5yr = t.NPI
    GROUP BY 1,2
),

-- ============================================
-- 6. MOST RECENT TIVI HCP (2YR)
-- ============================================

tivi_latest_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, FILL_DATE,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY FILL_DATE DESC, NPI) rn
    FROM tivi_tx_claims_2yr
),

tivi_latest_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS tivi_latest_tx_hcp, FILL_DATE
    FROM tivi_latest_tx_hcp_ranked
    WHERE rn = 1
),

tivi_latest_tx_stats AS (
    SELECT l.PATIENT_ID, l.tivi_latest_tx_hcp,
           COUNT(DISTINCT t.FILL_DATE) AS visit_count_5yr
    FROM tivi_latest_tx_hcp l
    LEFT JOIN tivi_tx_claims_5yr t
      ON l.PATIENT_ID = t.PATIENT_ID AND l.tivi_latest_tx_hcp = t.NPI
    GROUP BY 1,2
),

-- ============================================
-- 7. MOST SEEN (3YR)
-- ============================================

tivi_most_seen AS (
    SELECT PATIENT_ID, NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI
           ) rank
    FROM tivi_tx_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

tivi_most_seen_pivot AS (
    SELECT
        PATIENT_ID,
        MAX(CASE WHEN rank=1 THEN NPI END) AS tivi_most_seen_hcp1,
        MAX(CASE WHEN rank=2 THEN NPI END) AS tivi_most_seen_hcp2,
        MAX(CASE WHEN rank=3 THEN NPI END) AS tivi_most_seen_hcp3,
        MAX(CASE WHEN rank=4 THEN NPI END) AS tivi_most_seen_hcp4,
        MAX(CASE WHEN rank=5 THEN NPI END) AS tivi_most_seen_hcp5
    FROM tivi_most_seen
    GROUP BY PATIENT_ID
)

-- ============================================
-- FINAL SELECT
-- ============================================

SELECT
    ep.PATIENT_ID,

    ft.tivi_first_tx_hcp_5yr,
    fs.visit_count_5yr AS tivi_first_tx_visit_count_5yr,
    fs.last_visit_5yr AS tivi_last_visit_5yr,

    lt.tivi_latest_tx_hcp,
    ls.visit_count_5yr AS tivi_latest_tx_visit_count_5yr,

    mp.* EXCEPT (patient_id)

FROM eligible_patients ep

LEFT JOIN tivi_first_tx_hcp ft ON ep.PATIENT_ID = ft.PATIENT_ID
LEFT JOIN tivi_first_tx_stats fs ON ep.PATIENT_ID = fs.PATIENT_ID
LEFT JOIN tivi_latest_tx_hcp lt ON ep.PATIENT_ID = lt.PATIENT_ID
LEFT JOIN tivi_latest_tx_stats ls ON ep.PATIENT_ID = ls.PATIENT_ID
LEFT JOIN tivi_most_seen_pivot mp ON ep.PATIENT_ID = mp.PATIENT_ID;

In [0]:
%sql
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS

SELECT DISTINCT

    -- =====================================================
    -- PATIENT CORE
    -- =====================================================
    a.PATIENT_ID,
    a.PATIENT_YOB,
    a.PATIENT_AGE,
    a.PATIENT_GENDER,
    a.patient_state,

    -- =====================================================
    -- MILESTONES
    -- =====================================================
    a.incidence_date,
    a.first_incidence_treatment_date,
    a.latest_claim_date,
    a.latest_treatment_date,

    -- =====================================================
    -- ELAPRASE - FIRST DX / TX (5YR)
    -- =====================================================
    a.first_dx_hcp_5yr,
    a.first_dx_all_visit_count_5yr,
    a.first_dx_last_visit_5yr,

    a.first_tx_hcp_5yr,
    a.first_tx_all_visit_count_5yr,
    a.first_tx_last_visit_5yr,

    -- =====================================================
    -- ELAPRASE - MOST SEEN HCPs
    -- =====================================================
    a.most_seen_hcp1_3yr_ranked,
    a.most_seen_hcp1_visit_count_5yr,
    a.most_seen_hcp1_last_visit_5yr,

    a.most_seen_hcp2_3yr_ranked,
    a.most_seen_hcp2_visit_count_5yr,
    a.most_seen_hcp2_last_visit_5yr,

    a.most_seen_hcp3_3yr_ranked,
    a.most_seen_hcp3_visit_count_5yr,
    a.most_seen_hcp3_last_visit_5yr,

    a.most_seen_hcp4_3yr_ranked,
    a.most_seen_hcp4_visit_count_5yr,
    a.most_seen_hcp4_last_visit_5yr,

    a.most_seen_hcp5_3yr_ranked,
    a.most_seen_hcp5_visit_count_5yr,
    a.most_seen_hcp5_last_visit_5yr,

    -- =====================================================
    -- ELAPRASE - LATEST CLAIM HCP
    -- =====================================================
    a.latest_claim_hcp_npi,
    a.latest_claim_hcp_name,
    a.latest_claim_hcp_specialty,
    a.latest_claim_hcp_visit_count,
    a.latest_claim_hcp_hco_npi,
    a.latest_claim_hcp_hco_name,

    -- =====================================================
    -- ELAPRASE - LATEST TREATMENT HCP
    -- =====================================================
    a.latest_treatment_hcp_npi,
    a.latest_treatment_hcp_name,
    a.latest_treatment_hcp_specialty,
    a.latest_treatment_hcp_visit_count,
    a.latest_treatment_hcp_hco_npi,
    a.latest_treatment_hcp_hco_name,

    -- =====================================================
    -- MOST RECENTLY TREATED HCP (2YR)
    -- =====================================================
    b.* EXCEPT (patient_id),

    -- =====================================================
    -- PRIMARY HCP
    -- =====================================================
    c.* EXCEPT (patient_id),

    -- =====================================================
    -- =====================================================
    -- 🔵 TIVI CARE TEAM BLOCK
    -- =====================================================
    -- =====================================================

    -- =========================
    -- FIRST TIVI HCP (5YR)
    -- =========================
    d.tivi_first_tx_hcp_5yr,
    d.tivi_first_tx_visit_count_5yr,

    CONCAT(pdtivi_first_tx.FIRST_NAME, ' ', pdtivi_first_tx.LAST_NAME) AS tivi_first_tx_hcp_name_5yr,
    pdtivi_first_tx.PRIMARY_SPECIALTY AS tivi_first_tx_hcp_specialty_5yr,
    ref_tivi_first_tx.hco_npi AS tivi_first_tx_hcp_hco_npi_5yr,
    ref_tivi_first_tx.hco_name AS tivi_first_tx_hcp_hco_name_5yr,

    -- =========================
    -- LATEST TIVI HCP
    -- =========================
    d.tivi_latest_tx_hcp,
    d.tivi_latest_tx_visit_count_5yr,

    CONCAT(pdtivi_latest_tx.FIRST_NAME, ' ', pdtivi_latest_tx.LAST_NAME) AS tivi_latest_tx_hcp_name,
    pdtivi_latest_tx.PRIMARY_SPECIALTY AS tivi_latest_tx_hcp_specialty,
    ref_tivi_latest_tx.hco_npi AS tivi_latest_tx_hcp_hco_npi,
    ref_tivi_latest_tx.hco_name AS tivi_latest_tx_hcp_hco_name,

    -- =========================
    -- MOST SEEN TIVI HCPs
    -- =========================
    d.tivi_most_seen_hcp1,
    d.tivi_most_seen_hcp2,
    d.tivi_most_seen_hcp3,
    d.tivi_most_seen_hcp4,
    d.tivi_most_seen_hcp5

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base a

-- =====================================================
-- ELAPRASE ENRICHMENTS
-- =====================================================
LEFT JOIN most_recently_treated_hcp b
    ON a.patient_id = b.patient_id

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.primary_hcp c
    ON a.patient_id = c.patient_id

-- =====================================================
-- TIVI SUMMARY (PRE-AGGREGATED — SAFE JOIN)
-- =====================================================
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_tivi_hcp_summary d
    ON a.patient_id = d.patient_id

-- =====================================================
-- TIVI PROVIDER ENRICHMENT
-- =====================================================

-- FIRST TIVI
LEFT JOIN com_edp_prd.com_raw.kom_providers pdtivi_first_tx
    ON d.tivi_first_tx_hcp_5yr = pdtivi_first_tx.npi
   AND pdtivi_first_tx.provider_type = 'INDIVIDUAL'

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file ref_tivi_first_tx
    ON d.tivi_first_tx_hcp_5yr = ref_tivi_first_tx.hcp_npi

-- LATEST TIVI
LEFT JOIN com_edp_prd.com_raw.kom_providers pdtivi_latest_tx
    ON d.tivi_latest_tx_hcp = pdtivi_latest_tx.npi
   AND pdtivi_latest_tx.provider_type = 'INDIVIDUAL'

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file ref_tivi_latest_tx
    ON d.tivi_latest_tx_hcp = ref_tivi_latest_tx.hcp_npi;